# TMol How-To Guide — Constraints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/master/docs/tutorial/06_constraints.ipynb)

## 6. Constraints

Constraints are extra harmonic (or similar) restraint terms attached to a `PoseStack` and scored like any other energy term, at weight 0 by default. Section 2 already used one ready-made constraint generator; this notebook covers the general mechanism and a few more targeted examples.

### Setup

Wheel install, imports, seeding, and device selection live in [notebook_setup.py](https://github.com/uw-ipd/tmol/blob/master/docs/tutorial/notebook_setup.py).

In [1]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlretrieve

    urlretrieve(
        "https://raw.githubusercontent.com/uw-ipd/tmol/"
        "master/docs/tutorial/notebook_setup.py",
        "notebook_setup.py",
    )

from notebook_setup import *  # noqa: F401,F403

device, param_db = setup()

W1021 18:28:31.618000 93735 site-packages/torch/utils/cpp_extension.py:183] No CUDA runtime is found, using CUDA_HOME='/home/jflat06/miniconda3/envs/tmol'


### Generating constraints programmatically

Constraints can be added a pose_stack, with a function operating on up to 4 atoms and optional parameters.

First, `ConstraintSet.create_empty` creates the ConstraintSet, which constraint definitions are then added to.

Each constraint requires a function, a list of atoms (specified through the notation in the example below), and optional parameters. A number of built-in constraint functions are provided, but you can also define your own functions. 

After creating the ConstraintSet, apply it to the pose_stack with evolve, since PoseStack is immutable.

In [2]:
import attrs

cif_text = fetch_cif("1UBQ")
pose_stack = pose_stack_from_file(io.StringIO(cif_text), device, param_db=param_db)

constraint_atoms = tmol.constraint_atoms_for_labels(
    pose_stack, 
    [   # pose_ind, chain_id, residue_id (including insertion codes if necessary), atom_id
        (0, "A", "4", "C"), # atom 1
        (0, "A", "66", "N")  # atom 2
    ]
).unsqueeze(0)
constraint_params = torch.tensor([[1.47, 0.1]], dtype=torch.float32, device=device)

distance_constraints = ConstraintSet.create_empty(device=device, n_poses=pose_stack.n_poses).add_constraints(
    ConstraintEnergyTerm.harmonic, constraint_atoms, constraint_params
)
print(f"constraints defined: {distance_constraints.constraint_atoms.shape[0]}")

constrained_pose = attrs.evolve(pose_stack, constraint_set=distance_constraints)
view_constraints(constrained_pose)


constraints defined: 1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Scoring constraints

Once attached, it scores automatically through the ordinary `constraint` `ScoreType` -- no separate argument to scoring calls -- as long as that term's weight is nonzero.

In [3]:
score_function = beta2016_score_function(device, param_db=param_db)
score_function.set_weight(ScoreType.constraint, 1.0) # On by default, but not run if no constraints exist
scorer = score_function.render_whole_pose_scoring_module(constrained_pose)
weighted_terms = scorer(constrained_pose.coords, sum_terms=False)
constraint_index = score_function.all_score_types().index(ScoreType.constraint)
print(f"constraint energy: {float(weighted_terms[constraint_index, 0]):.3f}")

constraint energy: 1365.108


/home/jflat06/rosetta/tmol/tmol/score/constraint/_constraint_energy_term.py:469: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  sparse_scores = torch.sparse_coo_tensor(


### Common use cases with utility functions

#### Backbone-only coordinate constraints

`create_mainchain_coordinate_constraints` restrains every declared main-chain atom of every polymer block to its current coordinate; `constrain_all_ca` is the sparser version, one restraint per CA only.

In [4]:
mainchain_constrained = create_mainchain_coordinate_constraints(pose_stack.clone())
ca_only_constrained = constrain_all_ca(pose_stack.clone())
print(f"mainchain constraints: {mainchain_constrained.constraint_set.constraint_atoms.shape[0]}")
print(f"CA-only constraints:   {ca_only_constrained.constraint_set.constraint_atoms.shape[0]}")

view_constraints(ca_only_constrained)


mainchain constraints: 228
CA-only constraints:   76


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### PPI interface pairwise distance constraints

`constrain_closest_interchain_ca_pair(pose_stack, chain_a, chain_b)` finds the closest CA-CA pair across two chains, in each pose independently, and restrains it to its current distance -- a common way to hold an interface together while the rest of a structure is sampled or minimized.

In [5]:
interface_cif_text = fetch_cif("1AY7")
interface_pose_stack = pose_stack_from_file(io.StringIO(interface_cif_text), device, param_db=param_db)

ppi_constrained_pose = constrain_closest_interchain_ca_pair(interface_pose_stack, "A", "B")
print("PPI interface constraint attached")

closest_a, closest_b = ppi_constrained_pose.constraint_set.constraint_atoms[0, :2, 1].tolist()
current_distance = float(ppi_constrained_pose.constraint_set.constraint_params[0, 0])
print(
    f"closest CA-CA pair: {residue_label(interface_pose_stack, 0, closest_a)} and "
    f"{residue_label(interface_pose_stack, 0, closest_b)}, {current_distance:.2f} A apart"
)

view_constraints(ppi_constrained_pose)


LYS is seen only at a chain terminus and N, NZ could each carry its other connection; it is read as the conventional backbone through N. A copy of the residue in a chain would settle it.


PPI interface constraint attached
closest CA-CA pair: A 85 HIS and B 31 GLY, 4.81 A apart


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### Protein-ligand interface pairwise distance constraints

Same mechanism, between a protein atom and a ligand atom.

In [6]:
from tmol.io import atom_array_from_file, pose_stack_from_atom_array

pl_cif_text = fetch_cif("1STP")
pl_pose_stack = pose_stack_from_atom_array(
    atom_array_from_file(io.StringIO(pl_cif_text)), device, prepare_ligands=True, param_db=param_db
)

ligand_block = int(torch.nonzero(tmol.get_ligand_mask(pl_pose_stack)[0]).item())
protein_blocks = [
    block for block in range(pl_pose_stack.max_n_blocks)
    if pl_pose_stack.is_real_block(0, block) and block != ligand_block
]

ligand_type = pl_pose_stack.block_type(0, ligand_block)
ligand_offset = int(pl_pose_stack.block_coord_offset[0, ligand_block])
ligand_coords = pl_pose_stack.coords[0, ligand_offset : ligand_offset + len(ligand_type.atoms)]

best = None
for block in protein_blocks:
    block_type = pl_pose_stack.block_type(0, block)
    offset = int(pl_pose_stack.block_coord_offset[0, block])
    block_coords = pl_pose_stack.coords[0, offset : offset + len(block_type.atoms)]
    distances = torch.cdist(ligand_coords, block_coords)
    nearest = float(distances.min())
    if best is None or nearest < best[0]:
        flat_index = int(distances.argmin())
        ligand_atom, protein_atom = divmod(flat_index, distances.shape[1])
        best = (nearest, block, ligand_atom, protein_atom)

distance, closest_block, ligand_atom_index, protein_atom_index = best
closest_type = pl_pose_stack.block_type(0, closest_block)
print(
    f"closest ligand-protein contact: {ligand_type.atoms[ligand_atom_index].name} to "
    f"{residue_label(pl_pose_stack, 0, closest_block)} {closest_type.atoms[protein_atom_index].name}, "
    f"{distance:.2f} A"
)

pl_atoms = torch.zeros((1, 2, 3), dtype=torch.int32, device=device)
pl_atoms[0, 0] = torch.tensor([0, ligand_block, ligand_atom_index])
pl_atoms[0, 1] = torch.tensor([0, closest_block, protein_atom_index])
pl_params = torch.tensor([[distance, 0.5]], dtype=torch.float32, device=device)

pl_constraints = ConstraintSet.create_empty(device=device, n_poses=1).add_constraints(
    ConstraintEnergyTerm.harmonic, pl_atoms, pl_params
)
pl_constrained_pose = attrs.evolve(pl_pose_stack, constraint_set=pl_constraints)
print("protein-ligand interface constraint attached")

view_constraints(pl_constrained_pose)


*** Open Babel Warning  in WriteMolecule
  No 2D or 3D coordinates exist. Stereochemical information will be stored using an Open Babel extension. To generate 2D or 3D coordinates instead use --gen2D or --gen3D.


closest ligand-protein contact: O3 to A 43 TYR HH, 1.64 A
protein-ligand interface constraint attached


3Dmol.js failed to load for some reason. Please check your browser console for error messages.